# Auditable Wikipedia RAG-ETL Pipeline with FAISS IVF
## Presentation Notebook (Architecture + Policies)

> This notebook focuses on the **general pipeline flow and design policies**,
> and also includes a **minimal runnable full-build + retrieval** section for live query demo.
> For full run-story coverage (idempotency + augmented delta), use `notebooks/live_demo.ipynb`.


In [48]:
import os, sys, json, shlex, subprocess, importlib
from pathlib import Path


def ensure_module(module_name: str, pip_name: str | None = None):
    """Install missing modules on fresh environments (e.g., Google Colab)."""
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        target = pip_name or module_name
        print(f"[SETUP] Installing missing dependency: {target}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", target])


# Core deps needed by this notebook
ensure_module("duckdb", "duckdb")
ensure_module("pandas", "pandas")

import duckdb
import pandas as pd


def is_project_root(p: Path) -> bool:
    return (p / "src").is_dir() and (p / "data").is_dir()


def detect_project_root(start: Path) -> Path:
    current = start.resolve()

    # 1) Current directory + parents
    for candidate in [current, *current.parents]:
        if is_project_root(candidate):
            return candidate

    # 2) Common local + Colab locations
    home = Path.home()
    common_candidates = [
        home / "Desktop" / "455 Final",
        home / "455 Final",
        Path("/content") / "455 Final",
        Path("/content") / "drive" / "MyDrive" / "455 Final",
        current / "455 Final",
        current / "Desktop" / "455 Final",
    ]
    for candidate in common_candidates:
        if is_project_root(candidate):
            return candidate

    raise RuntimeError(
        "Could not locate project root containing 'src/' and 'data/'. "
        "Set cwd to repository root and rerun this cell."
    )


PROJECT_ROOT = detect_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

PYTHON       = sys.executable
LIVE_DB      = PROJECT_ROOT / "outputs/live_demo/wiki_demo.duckdb"
FULL_DB      = PROJECT_ROOT / "outputs/full/wiki.duckdb"
LIVE_INITIAL = PROJECT_ROOT / "data/live_demo/initial_sample.jsonl"
LIVE_UPDATE  = PROJECT_ROOT / "data/live_demo/update_sample.jsonl"
LIVE_INDEX   = PROJECT_ROOT / "outputs/live_demo/faiss/index.faiss"
FULL_INDEX   = PROJECT_ROOT / "outputs/full/faiss/index.faiss"
SRC_DIR      = PROJECT_ROOT / "src"

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# ── run_cmd: run a shell command from project root ──────────
def run_cmd(cmd):
    parts = shlex.split(cmd)
    if parts[0] == "python":
        parts[0] = PYTHON
    print(f"$ {' '.join(str(p) for p in parts)}")
    print("─" * 70)
    subprocess.run(parts, cwd=str(PROJECT_ROOT))
    print("─" * 70)

# ── connect_db: open DuckDB read-only (None if missing) ─────
def connect_db(db_path):
    if not Path(db_path).exists():
        print(f"Database not found: {db_path}")
        return None
    return duckdb.connect(str(db_path), read_only=True)

# ── query_df: SQL → DataFrame with error guard ──────────────
def query_df(con, sql):
    try:
        return con.execute(sql).df()
    except Exception as e:
        print(f"Query error: {e}")
        return pd.DataFrame()

# ── show_table: display a table if it exists ────────────────
def show_table(con, table_name, limit=10):
    if con is None:
        print("No database connection.")
        return
    try:
        tables = [r[0] for r in con.execute("SHOW TABLES").fetchall()]
        if table_name not in tables:
            print(f"Table not found: {table_name}")
            return
        df = con.execute(f"SELECT * FROM {table_name} LIMIT {limit}").df()
        display(df)
    except Exception as e:
        print(f"Error showing '{table_name}': {e}")

# ── code_excerpt: print lines around a search string ────────
def code_excerpt(path, search, before=10, after=25):
    p = PROJECT_ROOT / path
    if not p.exists():
        print(f"File not found: {path}")
        return
    lines = p.read_text().splitlines()
    idx = next((i for i, ln in enumerate(lines) if search in ln), None)
    if idx is None:
        print(f"Search term not found: '{search}' in {p.name}")
        return
    s = max(0, idx - before)
    e = min(len(lines), idx + after)
    print(f"# {p.name}  (lines {s+1}–{e})")
    print("─" * 60)
    for i in range(s, e):
        marker = "►" if i == idx else " "
        print(f"{marker} {i+1:4d}│ {lines[i]}")

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {PYTHON}")


Project root : C:\Users\Gawon Lim\Desktop\455 Final
Python       : c:\Users\Gawon Lim\anaconda3\envs\RAG\python.exe


---
# 1 · Project Framing

> **"My project is an auditable ETL pipeline that incrementally maintains a Wikipedia-based
> FAISS IVF retrieval system. The goal is not only to retrieve relevant Wikipedia chunks,
> but to show how the data system behaves when documents are added, updated, duplicated,
> or corrupted."**

> **This is an ETL project first, and a RAG system second.**

---

### End-to-end data flow

```
Raw Wikipedia JSON
    → [INGEST]   raw_documents          (staging — verbatim copy)
    → [CLEAN]    rejected_documents     (staging — bad rows quarantined)
    → [CURATE]   clean_documents        (curation — SCD Type 2 versioning)
    → [CHUNK]    fact_chunks            (curation — 384-token sliding window)
    → [EMBED]    fact_embeddings        (curation — E5-base-v2, normalised)
    → [INDEX]    FAISS IVF index        (serving — derived from active embeddings)
    → [QUERY]    Retrieval results      (serving — resolved back through DuckDB)
    → [MONITOR]  runs / row_count_reconciliation / audit_results / latency_logs
```

Two boundaries matter:
- **Staging** (`raw_documents`, `rejected_documents`) — append-only, never modified after insert.
- **Curation** (`clean_documents`, `fact_chunks`, `fact_embeddings`) — SCD Type 2,
  exactly one active version per document at any time.

---
# 2 · Rubric Requirements & Execution Tracks

| Requirement | Implementation |
|-------------|---------------|
| One-command full rebuild | `src/build.py --mode full` |
| Augmented / incremental build | `src/build.py --mode augmented` |
| Idempotency with audit artifacts | `src/audit.py` → `audit_results` table |
| Monitoring evidence | `runs`, `row_count_reconciliation`, `latency_logs` |
| Failure catalog | `rejected_documents` with `reason_code` |
| FAISS IVF retrieval (AI bonus) | `src/index_ivf.py` + `src/retrieve.py` |

### Two execution tracks

| Track | Dataset | Speed | Purpose |
|-------|---------|-------|---------|
| **Full offline** | `data/full/raw_wiki.jsonl` | Slow (run before presentation) | Demonstrates scale |
| **Live demo** | `data/live_demo/initial_sample.jsonl` (40 docs) | Fast (runs live) | Demonstrates all ETL behaviors |

Both tracks use **identical source code**, **identical schema**, and **identical pipeline logic**.

In [49]:
# Verify that live demo files exist
for label, path in [
    ("LIVE_INITIAL", LIVE_INITIAL),
    ("LIVE_UPDATE",  LIVE_UPDATE),
    ("LIVE_DB",      LIVE_DB),
    ("LIVE_INDEX",   LIVE_INDEX),
    ("FULL_DB",      FULL_DB),
    ("FULL_INDEX",   FULL_INDEX),
]:
    status = "✓" if Path(path).exists() else "✗ MISSING"
    print(f"  {status}  {label:14s}  {path}")


  ✓  LIVE_INITIAL    C:\Users\Gawon Lim\Desktop\455 Final\data\live_demo\initial_sample.jsonl
  ✓  LIVE_UPDATE     C:\Users\Gawon Lim\Desktop\455 Final\data\live_demo\update_sample.jsonl
  ✓  LIVE_DB         C:\Users\Gawon Lim\Desktop\455 Final\outputs\live_demo\wiki_demo.duckdb
  ✓  LIVE_INDEX      C:\Users\Gawon Lim\Desktop\455 Final\outputs\live_demo\faiss\index.faiss
  ✓  FULL_DB         C:\Users\Gawon Lim\Desktop\455 Final\outputs\full\wiki.duckdb
  ✓  FULL_INDEX      C:\Users\Gawon Lim\Desktop\455 Final\outputs\full\faiss\index.faiss


---
# 3 · Architecture Overview

> **"DuckDB is the system of record for document lifecycle, chunks, metadata,
> monitoring, and audit artifacts.
> FAISS is a derived serving index built from the active chunk embeddings."**

### Source modules

| Module | Responsibility |
|--------|---------------|
| `src/ingest.py` | JSONL → `raw_documents` |
| `src/clean.py` | Validate, clean text, quarantine bad rows → `rejected_documents` |
| `src/curate.py` | SCD Type 2: NEW / UPDATED / UNCHANGED → `clean_documents` |
| `src/chunk.py` | Sliding-window tokenisation → `fact_chunks` |
| `src/embed.py` | E5-base-v2 encoding → `fact_embeddings` + `.npy` files |
| `src/index_ivf.py` | Train + build IVFFlat → `faiss_index_registry` + `index.faiss` |
| `src/retrieve.py` | Embed query → FAISS search → resolve chunks via DuckDB |
| `src/audit.py` | Cross-run idempotency checks → `audit_results` |
| `src/monitoring.py` | Schema init, run lifecycle, reconciliation, latency logs |

In [50]:
# List all source modules
print("src/ modules:")
for f in sorted(SRC_DIR.glob("*.py")):
    print(f"  {f.name}")


src/ modules:
  __init__.py
  audit.py
  build.py
  chunk.py
  clean.py
  config.py
  curate.py
  download_wiki.py
  embed.py
  index_ivf.py
  ingest.py
  make_live_demo_data.py
  monitoring.py
  retrieve.py
  utils.py


In [51]:
# Orchestration entry point — [STAGE] labels show pipeline order
code_excerpt("src/build.py", "[STAGE] INGEST", before=2, after=40)


# build.py  (lines 57–98)
────────────────────────────────────────────────────────────
    57│         # ---- STAGE: INGEST ----
    58│         t0 = now_utc()
►   59│         print(f"\n[STAGE] INGEST")
    60│         raw_count = ingest.ingest_raw_documents(conn, input_path, run_id)
    61│         monitoring.log_latency(conn, run_id, "ingest", t0, now_utc(), raw_count)
    62│         monitoring.update_run_counts(conn, run_id, raw_doc_count=raw_count)
    63│ 
    64│         # ---- STAGE: CLEAN ----
    65│         t0 = now_utc()
    66│         print(f"\n[STAGE] CLEAN")
    67│         valid_docs, rejected_count = clean.validate_and_clean_raw(conn, run_id)
    68│         monitoring.log_latency(conn, run_id, "clean", t0, now_utc(), len(valid_docs))
    69│         monitoring.update_run_counts(
    70│             conn, run_id,
    71│             stg_valid_count=len(valid_docs),
    72│             rejected_doc_count=rejected_count
    73│         )
    74│ 
    75│         # ---- 

---
# 4 · Data Grain and DuckDB Schema

| Table | Grain |
|-------|-------|
| `raw_documents` | one row = one raw Wikipedia document per run |
| `clean_documents` | one row = one cleaned document **version** |
| `fact_chunks` | one row = one E5-tokenised chunk from one active document version |
| `fact_embeddings` | one row = one chunk embedding (vectors stored in `.npy` files) |

> **The main retrieval fact table is `fact_chunks`.
> Core grain: one chunk from one active document version.**

Supporting tables: `runs`, `rejected_documents`, `row_count_reconciliation`,
`audit_results`, `faiss_index_registry`, `retrieval_eval`, `latency_logs`.

In [52]:
con = connect_db(FULL_DB)
if con:
    print("DuckDB tables in wiki_demo.duckdb:")
    for row in con.execute("SHOW TABLES").fetchall():
        cnt = con.execute(f"SELECT COUNT(*) FROM {row[0]}").fetchone()[0]
        print(f"  {row[0]:<35s}  {cnt:>6d} rows")
    con.close()


DuckDB tables in wiki_demo.duckdb:
  audit_results                             5 rows
  clean_documents                        2000 rows
  fact_chunks                           30008 rows
  fact_embeddings                       30008 rows
  faiss_index_registry                      2 rows
  latency_logs                             21 rows
  raw_documents                          9585 rows
  rejected_documents                        0 rows
  retrieval_eval                            0 rows
  row_count_reconciliation                  2 rows
  runs                                      5 rows


In [53]:
# Schema created in src/monitoring.py (search: "SCHEMA_SQL")
code_excerpt("src/monitoring.py", "SCHEMA_SQL", before=0, after=40)


# monitoring.py  (lines 10–49)
────────────────────────────────────────────────────────────
►   10│ SCHEMA_SQL = """
    11│ CREATE SEQUENCE IF NOT EXISTS run_id_seq START 1;
    12│ 
    13│ CREATE TABLE IF NOT EXISTS runs (
    14│     run_id INTEGER PRIMARY KEY,
    15│     mode VARCHAR,
    16│     dataset_name VARCHAR,
    17│     input_path VARCHAR,
    18│     output_dir VARCHAR,
    19│     start_time TIMESTAMP,
    20│     end_time TIMESTAMP,
    21│     status VARCHAR,
    22│     raw_doc_count INTEGER DEFAULT 0,
    23│     stg_valid_count INTEGER DEFAULT 0,
    24│     rejected_doc_count INTEGER DEFAULT 0,
    25│     new_doc_count INTEGER DEFAULT 0,
    26│     updated_doc_count INTEGER DEFAULT 0,
    27│     duplicate_doc_count INTEGER DEFAULT 0,
    28│     unchanged_doc_count INTEGER DEFAULT 0,
    29│     active_doc_count INTEGER DEFAULT 0,
    30│     new_chunk_count INTEGER DEFAULT 0,
    31│     active_chunk_count INTEGER DEFAULT 0,
    32│     new_embedding_count I

In [54]:
con = connect_db(LIVE_DB)
if con:
    print("── raw_documents columns ──")
    display(con.execute("DESCRIBE raw_documents").df())
    print("\n── clean_documents columns ──")
    display(con.execute("DESCRIBE clean_documents").df())
    print("\n── fact_chunks columns ──")
    display(con.execute("DESCRIBE fact_chunks").df())
    con.close()


── raw_documents columns ──


,column_name,column_type,null,key,default,extra
0,run_id,INTEGER,YES,None,None,None
1,ingest_batch_id,VARCHAR,YES,None,None,None
2,id,VARCHAR,YES,None,None,None
3,url,VARCHAR,YES,None,None,None
4,title,VARCHAR,YES,None,None,None
5,text,VARCHAR,YES,None,None,None
6,ingested_at,TIMESTAMP,YES,None,None,None



── clean_documents columns ──


,column_name,column_type,null,key,default,extra
0,doc_id,VARCHAR,YES,None,None,None
1,url,VARCHAR,YES,None,None,None
2,title,VARCHAR,YES,None,None,None
3,cleaned_text,VARCHAR,YES,None,None,None
4,content_hash,VARCHAR,YES,None,None,None
5,text_len,INTEGER,YES,None,None,None
6,source_run_id,INTEGER,YES,None,None,None
7,is_active,BOOLEAN,YES,None,None,None
8,valid_from_run_id,INTEGER,YES,None,None,None
9,valid_to_run_id,INTEGER,YES,None,None,None



── fact_chunks columns ──


,column_name,column_type,null,key,default,extra
0,chunk_id,VARCHAR,YES,None,None,None
1,doc_id,VARCHAR,YES,None,None,None
2,content_hash,VARCHAR,YES,None,None,None
3,chunk_index,INTEGER,YES,None,None,None
4,start_tok,INTEGER,YES,None,None,None
5,end_tok,INTEGER,YES,None,None,None
6,chunk_text,VARCHAR,YES,None,None,None
7,chunk_token_len,INTEGER,YES,None,None,None
8,chunk_hash,VARCHAR,YES,None,None,None
9,source_run_id,INTEGER,YES,None,None,None


---
# 5 · Cleaning and Update Contract

## Cleaning policy implemented in `src/clean.py`

The cleaning function is intentionally deterministic and minimal so version detection is stable:

1. **Null-safe input**
   - If `text is None`, treat it as empty (`""`).

2. **Byte/whitespace normalization (`clean_text`)**
   - Remove null bytes: `text.replace("\\x00", "")`
   - Strip leading/trailing spaces: `text.strip()`
   - Collapse all repeated whitespace (spaces/newlines/tabs) to one space:
     `re.sub(r"\\s+", " ", text)`

3. **Validation and quarantine (`validate_and_clean_raw`)**
   - Reject if `id` is missing/blank -> `reason_code='MISSING_ID'`
   - Reject if raw text is empty after strip -> `reason_code='EMPTY_TEXT'`
   - Reject if text becomes empty after cleaning -> `reason_code='EMPTY_TEXT'`
   - Rejected rows are inserted into `rejected_documents` with a short `raw_snapshot`
     for auditability.

4. **Deterministic version key**
   - For valid rows, compute `content_hash = sha256(cleaned_text)`.
   - This hash is the update detector used by curation.

## Update policy (curation contract)

| Incoming doc vs current active version | Action |
|-------------|--------|
| New `doc_id` | **NEW** -> insert active row, then chunk + embed |
| Same `doc_id` + same `content_hash` | **UNCHANGED** -> skip downstream recomputation |
| Same `doc_id` + different `content_hash` | **UPDATED** -> deactivate old version, insert new active version, re-chunk/re-embed only affected doc |
| Invalid row (missing id / empty text) | **REJECTED** -> quarantined in `rejected_documents`, excluded from curated/index |

> Exactly one active version per `doc_id` is maintained.
> Duplicate submissions do not create new chunks/embeddings.


In [55]:
# Preview initial sample and update batch
print(f"initial_sample.jsonl  ({LIVE_INITIAL}):")
if LIVE_INITIAL.exists():
    docs = [json.loads(l) for l in LIVE_INITIAL.read_text().splitlines() if l.strip()]
    df_init = pd.DataFrame([{"id": d["id"], "title": d["title"],
                              "text_len": len(d.get("text",""))} for d in docs])
    display(df_init.head(5))
    print(f"  total: {len(docs)} documents")

print(f"\nupdate_sample.jsonl  ({LIVE_UPDATE}):")
if LIVE_UPDATE.exists():
    udocs = [json.loads(l) for l in LIVE_UPDATE.read_text().splitlines() if l.strip()]
    df_upd = pd.DataFrame([{"id": d["id"], "title": d.get("title",""),
                             "text_len": len(d.get("text") or ""),
                             "has_text": bool((d.get("text") or "").strip())}
                           for d in udocs])
    display(df_upd)
    print("  (corrupted doc = has_text: False)")


initial_sample.jsonl  (C:\Users\Gawon Lim\Desktop\455 Final\data\live_demo\initial_sample.jsonl):


,id,title,text_len
0,1260,Advanced Encryption Standard,21331
1,664,Astronaut,28312
2,334,International Atomic Time,7487
3,1366,Amethyst,11666
4,856,Apple Inc.,83378


  total: 40 documents

update_sample.jsonl  (C:\Users\Gawon Lim\Desktop\455 Final\data\live_demo\update_sample.jsonl):


,id,title,text_len,has_text
0,12,Anarchism,46064,True
1,1260,Advanced Encryption Standard,21424,True
2,664,Astronaut,28312,True
3,corrupted_doc_001,Corrupted Test Document,0,False


  (corrupted doc = has_text: False)


In [56]:
con = connect_db(LIVE_DB)
if con:
    print("── clean_documents (active versions) ──")
    display(query_df(con, '''
        SELECT doc_id, title,
               content_hash[:12] || '...' AS hash_prefix,
               is_active, valid_from_run_id, valid_to_run_id
        FROM clean_documents
        ORDER BY doc_id
        LIMIT 6
    '''))

    print("\n── rejected_documents (latest run that actually has rejects) ──")
    df_rej = query_df(con, '''
        SELECT r.run_id, r.mode, r.input_path,
               d.doc_id, d.reason_code, d.reason_detail
        FROM runs r
        JOIN rejected_documents d
          ON r.run_id = d.run_id
        WHERE r.run_id = (
            SELECT MAX(run_id)
            FROM runs
            WHERE rejected_doc_count > 0
        )
        ORDER BY d.doc_id
    ''')

    if df_rej.empty:
        print("No rejected rows found in this DB yet.")
        display(query_df(con, '''
            SELECT run_id, mode, input_path, rejected_doc_count
            FROM runs
            ORDER BY run_id DESC
            LIMIT 5
        '''))
    else:
        display(df_rej)

    con.close()


── clean_documents (active versions) ──


,doc_id,title,hash_prefix,is_active,valid_from_run_id,valid_to_run_id
0,1012,August 22,030d53f2e483...,True,1,<NA>
1,1014,Alcohol (chemistry),be95feadd98b...,True,1,<NA>
2,1032,Abscess,5a6c5c190db0...,True,1,<NA>
3,1111,Politics of American Samoa,83f9b6f6b9d5...,True,1,<NA>
4,1166,Afro Celt Sound System,4d49fe95d6a9...,True,1,<NA>
5,1176,Antisymmetric relation,afaacc3519a3...,True,1,<NA>



── rejected_documents (latest run that actually has rejects) ──
No rejected rows found in this DB yet.


,run_id,mode,input_path,rejected_doc_count
0,2,full,data\live_demo\initial_sample.jsonl,0
1,1,full,data\live_demo\initial_sample.jsonl,0


In [33]:
# validate_and_clean_raw: validates & computes content_hash
code_excerpt("src/clean.py", "validate_and_clean_raw", before=0, after=30)


# clean.py  (lines 27–56)
────────────────────────────────────────────────────────────
►   27│ def validate_and_clean_raw(conn, run_id: int) -> tuple:
    28│     """
    29│     SELECT id, url, title, text FROM raw_documents WHERE run_id=?
    30│     Validate and clean each row.
    31│     Returns (list of valid_doc dicts, rejected_count).
    32│     """
    33│     rows = conn.execute(
    34│         "SELECT id, url, title, text FROM raw_documents WHERE run_id=?",
    35│         [run_id]
    36│     ).fetchall()
    37│ 
    38│     valid_docs = []
    39│     rejected_rows = []
    40│ 
    41│     for row in rows:
    42│         doc_id, url, title, text = row[0], row[1], row[2], row[3]
    43│ 
    44│         raw_snapshot = json.dumps({
    45│             "id": doc_id,
    46│             "url": url,
    47│             "title": title,
    48│             "text": text[:500] if text else None
    49│         })
    50│ 
    51│         # Validation: missing id
    52│       

---
# 6 · Chunking, E5-base-v2, and FAISS IVF Design

### Chunking policy

| Parameter | Value |
|-----------|-------|
| Input text | `"{title}: {cleaned_text}"` |
| Tokenizer | `intfloat/e5-base-v2` tokenizer |
| Chunk size | 384 tokens |
| Overlap | 64 tokens |
| Stride | 320 tokens |
| Min chunk length | 32 tokens |
| Chunk ID | `sha256(doc_id | content_hash | start_tok | end_tok)` (deterministic) |

Each chunk stores lineage (`doc_id`, `content_hash`, token boundaries), so every vector can be traced back to a specific document version.

### What is E5-base-v2?

`intfloat/e5-base-v2` is a sentence-transformer embedding model that maps text into dense vectors where semantic similarity corresponds to vector similarity.

- **Dual-prefix design**: passages are embedded with `"passage: "`, queries with `"query: "`.
  This helps align retrieval behavior between indexed chunks and user questions.
- **Embedding dimension**: 768-dimensional dense vectors.
- **Normalization**: vectors are L2-normalized (`normalize_embeddings=True`) so inner product behaves like cosine similarity.
- **Why this model here**: strong retrieval quality for short-to-medium passages, stable open-source deployment, and easy integration with SentenceTransformers.

### What is FAISS?

FAISS (Facebook AI Similarity Search) is a vector search library for approximate nearest-neighbor retrieval over dense embeddings.

In this project we use **FAISS `IndexIVFFlat`**:
- IVF = Inverted File index: partitions vector space into `nlist` clusters (coarse centroids).
- Query-time search probes only selected partitions (`nprobe`) for speed/recall tradeoff.
- `Flat` means exact distance inside each probed partition.

Pipeline contract around FAISS:
- FAISS index is a **derived serving artifact** from active embeddings.
- Full build: train/rebuild IVF from scratch.
- Augmented ETL: only changed docs are re-cleaned/chunked/embedded, then IVF is rebuilt from all active embeddings so centroids are trained on the current distribution.
- Provenance is tracked in `faiss_index_registry` (vector count, checksum, model, index params).


In [35]:
con = connect_db(LIVE_DB)
if con:
    n = con.execute("SELECT COUNT(*) FROM fact_chunks WHERE is_active=TRUE").fetchone()[0]
    print(f"Active fact_chunks: {n}")
    print()
    display(query_df(con, '''
        SELECT chunk_id[:12]||'...' AS chunk_id,
               doc_id, chunk_index, start_tok, end_tok,
               chunk_token_len,
               LEFT(chunk_text, 60) || '...' AS preview,
               is_active
        FROM fact_chunks
        WHERE is_active=TRUE
        ORDER BY doc_id, chunk_index
        LIMIT 5
    '''))

    print("\n── faiss_index_registry ──")
    show_table(con, "faiss_index_registry")
    con.close()


Active fact_chunks: 613



,chunk_id,doc_id,chunk_index,start_tok,end_tok,chunk_token_len,preview,is_active
0,afe38c4aba4c...,1012,0,0,384,384,august 22 : events pre - 1600 392 – arbogast has eugenius el...,True
1,764682ddb718...,1012,1,320,704,384,"saint - domingue, haiti. 1798 – french troops land at kilcum...",True
2,2391c16a3f53...,1012,2,640,1024,384,italy. 1944 – world war ii : holocaust of kedros in crete by...,True
3,1193d72dde7a...,1012,3,960,1344,384,suffers an engine fire during takeoff at manchester airport....,True
4,29a07edaec57...,1012,4,1280,1664,384,". 1464 ) 1570 – franz von dietrichstein, roman catholic arch...",True



── faiss_index_registry ──


,index_id,run_id,dataset_name,index_type,embedding_model,metric,nlist,nprobe,num_vectors,index_path,vector_map_path,index_checksum,build_mode,is_active,created_at
0,1908346e-d20b-4d98-b512-7117ff3622d3,1,live_demo,IVFFlat,intfloat/e5-base-v2,INNER_PRODUCT,61,10,613,outputs\live_demo\faiss\index.faiss,outputs\live_demo\faiss\vector_map.npy,74c7e8cfe0899dbdebdac1d7c6da56faba057aa254f15a680f5ba3257a1840e1,full,False,2026-05-03 23:20:02.081233
1,063a09b7-b76e-4380-b547-fcd625f2db53,2,live_demo,IVFFlat,intfloat/e5-base-v2,INNER_PRODUCT,61,10,613,outputs\live_demo\faiss\index.faiss,outputs\live_demo\faiss\vector_map.npy,74c7e8cfe0899dbdebdac1d7c6da56faba057aa254f15a680f5ba3257a1840e1,full,True,2026-05-03 23:20:12.761303


---
# 7 · Use Existing Full Artifacts (No Rebuild)

This section **reuses the full-corpus artifacts already in your folder**:
- `outputs/full/wiki.duckdb`
- `outputs/full/faiss/index.faiss`

Default flow here is: verify artifacts -> run retrieval -> show latest run summary.

(An optional rebuild command is kept in a separate cell only for fallback.)


In [57]:
# Verify existing full artifacts (reuse mode)
FULL_INPUT = PROJECT_ROOT / "data/full/raw_wiki.jsonl"
print("FULL_DB    :", FULL_DB, "exists:", FULL_DB.exists())
print("FULL_INDEX :", FULL_INDEX, "exists:", FULL_INDEX.exists())
print("FULL_INPUT :", FULL_INPUT, "exists:", FULL_INPUT.exists(), "(only needed for rebuild)")


FULL_DB    : C:\Users\Gawon Lim\Desktop\455 Final\outputs\full\wiki.duckdb exists: True
FULL_INDEX : C:\Users\Gawon Lim\Desktop\455 Final\outputs\full\faiss\index.faiss exists: True
FULL_INPUT : C:\Users\Gawon Lim\Desktop\455 Final\data\full\raw_wiki.jsonl exists: True (only needed for rebuild)


# Retrieval demo using existing full artifacts

In [61]:
# GPT comparison: baseline answer vs retrieval-grounded answer
import importlib

# Install openai on-the-fly if missing (fresh runtime safety)
try:
    importlib.import_module("openai")
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openai"])

from openai import OpenAI
from src.retrieve import retrieve

QUERY = "What is artificial intelligence?"
MODEL = "gpt-4o-mini"  # change if you want a different OpenAI model
TOP_K = 5

api_key_path = PROJECT_ROOT / "gpt.txt"
if not api_key_path.exists():
    raise FileNotFoundError(f"Missing API key file: {api_key_path}")

api_key = api_key_path.read_text(encoding="utf-8").strip()
if not api_key:
    raise ValueError("gpt.txt is empty. Put your OpenAI API key in that file.")

client = OpenAI(api_key=api_key)

# 1) Baseline (no retrieved context)
baseline = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": QUERY},
    ],
    temperature=0.2,
)
baseline_answer = baseline.choices[0].message.content

# 2) Retrieval-grounded answer using your own index
df_ctx = retrieve(
    query=QUERY,
    db_path=os.path.relpath(FULL_DB, PROJECT_ROOT),
    index_path=os.path.relpath(FULL_INDEX, PROJECT_ROOT),
    top_k=TOP_K,
)

if df_ctx.empty:
    raise RuntimeError("No retrieval results found. Check FULL_DB/FULL_INDEX first.")

context_blocks = []
for i, row in df_ctx.iterrows():
    context_blocks.append(
        f"[Doc {i+1}] title={row['title']} url={row['url']}\n{row['chunk_text']}"
    )
context_text = "\n\n".join(context_blocks)

rag_prompt = (
    f"Question: {QUERY}\n\n"
    "Answer based on these documents only. If uncertain, say what is missing.\n\n"
    f"Documents:\n{context_text}"
)

rag = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a careful assistant that grounds claims in provided documents."},
        {"role": "user", "content": rag_prompt},
    ],
    temperature=0.2,
)
rag_answer = rag.choices[0].message.content

print("=== Question ===")
print(QUERY)
print("\n=== Baseline GPT Answer (no retrieval) ===")
print(baseline_answer)
print("\n=== Retrieval-Grounded GPT Answer ===")
print(rag_answer)

print("\n=== Retrieved Evidence Used (top-k) ===")
display(df_ctx[["rank", "score", "title", "url", "chunk_text"]])

c:\Users\Gawon Lim\anaconda3\envs\RAG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7654.12it/s]


=== Question ===
What is artificial intelligence?

=== Baseline GPT Answer (no retrieval) ===
Artificial intelligence (AI) refers to the simulation of human intelligence processes by computer systems. These processes include learning (the acquisition of information and rules for using it), reasoning (using rules to reach approximate or definite conclusions), and self-correction. AI can be categorized into several types, including:

1. **Narrow AI**: Also known as weak AI, this type of AI is designed to perform a specific task, such as facial recognition, language translation, or playing a game. Most of the AI applications in use today fall into this category.

2. **General AI**: Also known as strong AI or AGI (Artificial General Intelligence), this type of AI would have the ability to understand, learn, and apply intelligence across a wide range of tasks, similar to a human being. As of now, general AI remains largely theoretical and has not yet been achieved.

3. **Machine Learning**:

,rank,score,title,url,chunk_text
0,1,0.863636,Artificial intelligence,https://en.wikipedia.org/wiki/Artificial%20intelligence,artificial intelligence : artificial intelligence ( ai ) is the intelligence...
1,2,0.858490,Ai,https://en.wikipedia.org/wiki/Ai,"ai : ai is artificial intelligence, intellectual ability in machines and rob..."
2,3,0.823815,Ai,https://en.wikipedia.org/wiki/Ai,"institute, an association of real estate appraisers the art institutes, a ch..."
3,4,0.822608,Artificial intelligence,https://en.wikipedia.org/wiki/Artificial%20intelligence,". ai also draws upon psychology, linguistics, philosophy, neuroscience and m..."
4,5,0.819140,Artificial intelligence,https://en.wikipedia.org/wiki/Artificial%20intelligence,"will change the state in a particular way, and a reward function that suppli..."


In [60]:
if FULL_DB.exists() and FULL_INDEX.exists():
    run_cmd(
        'python src/retrieve.py '
        '--db outputs/full/wiki.duckdb '
        '--index outputs/full/faiss/index.faiss '
        '--query "What is artificial intelligence?" '
        '--top-k 5'
    )
else:
    print("Full artifacts missing. Check Section 7 artifact verification cell first.")

$ c:\Users\Gawon Lim\anaconda3\envs\RAG\python.exe src/retrieve.py --db outputs/full/wiki.duckdb --index outputs/full/faiss/index.faiss --query What is artificial intelligence? --top-k 5
──────────────────────────────────────────────────────────────────────
──────────────────────────────────────────────────────────────────────


src/retrieve.py --db outputs/full/wiki.duckdb --index outputs/full/faiss/index.faiss --query What is artificial intelligence? --top-k 5